In [1]:
import pandas as pd
import numpy as np
import re
import pdb
from collections import defaultdict
import os

In [ ]:
# Einlesen der Date aus dem Notebook "1_PuRe Rohdatenextrakt.jpynb"
df_pure = pd.read_excel('Daten/df_pure.xlsx')

# Alternativ kann ich auch direkt die Daten aus dem json-download über Openrefine einlesen
#df_pure = pd.read_excel('download-json.xlsx')

# Zur Sicherheit die Anzahl an IDs = Datensätzen zählen
df_pure['90999_ItemID'].nunique() 

447

In [ ]:
# Auskommentierung aufheben, um es sich anzusehen
# df_pure

,90999_ItemID,data - versionNumber,data - modificationDate,data - versionState,data - versionPid,data - lastModificationDate,data - publicState,data - objectPid,data - creationDate,data - message,...,data - latestVersion - objectId,data - latestVersion - versionNumber,data - latestVersion - modificationDate,data - latestVersion - versionState,data - latestVersion - versionPid,data - latestVersion - modifier - objectId,Item_seq,Name,kleinster_wert,1103_Ersterscheinungsdatum
0,item_3699285,2.0,2026-04-23T12:29:32.939+00:00,RELEASED,hdl:21.11116/0000-0012-FA72-2,2026-04-23T12:29:32.939+00:00,RELEASED,hdl:21.11116/0000-0012-BC65-7,2026-03-25T10:54:08.925+00:00,NaN,...,item_3699285,2.0,2026-04-23T12:29:32.939+00:00,RELEASED,hdl:21.11116/0000-0012-FA72-2,user_1241569,1,Luca Cigna,2026.0,2026.0
1,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN
2,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN
3,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4,Nina Lopez-Uroz,NaN,NaN
4,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2332,item_3458140,2.0,2022-11-09T13:27:47.664+00:00,RELEASED,hdl:21.11116/0000-000B-6898-5,2022-11-09T13:27:47.664+00:00,RELEASED,hdl:21.11116/0000-000B-5C0E-0,2022-11-02T13:32:56.911+00:00,NaN,...,item_3458140,2.0,2022-11-09T13:27:47.664+00:00,RELEASED,hdl:21.11116/0000-000B-6898-5,user_1241569,1,Vincenzo Maccarrone,2022.0,2022.0
2333,item_3458140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2,Arianna Tassinari,NaN,NaN
2334,item_3458140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN
2335,item_3458140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN,NaN


In [102]:
# Umbenennung der hier relevanten Spalten
df_pure = df_pure.rename(columns={
    'data - metadata - title':'1001_Titel',
    'data - metadata - genre':'1002_Genre',
    'data - metadata - degree' : '1021_Degree',
    'data - metadata - sources - _ - title':'1101_Quellentitel',
    'data - metadata - reviewMethod':'1108_Review',
    '1103_Ersterscheinungsdatum':'Datum',
    'data - files - _ - content':'4_File-content',
    'data - files - _ - metadata - title':'2_File-title',
    'data - files - _ - visibility':'5_File-visibility',
    'data - files - _ - storage':'7_File-Storage',
    'data - files - _ - metadata - contentCategory':'3_File-Category',
    'data - files - _ - metadata - description':'1_File-Description',
    'data - files - _ - metadata - oaStatus':'6_File-OA',
    'data - files - _ - metadata - license':'8_File-Licence',
    'data - files - _ - metadata - embargoUntil':'9_File-Embargo',
    'data - objectId':'90999_ItemID',}) 




In [103]:
# Auswahl der Spalten und Übertrag in ein neues df

cropped_df = df_pure[['1001_Titel',
                      '1002_Genre',
                      'Datum',
                      '1101_Quellentitel',
                      '1_File-Description',
                      '2_File-title',
                      '3_File-Category',
                      '4_File-content',
                      '5_File-visibility',
                      '6_File-OA', 
                      '7_File-Storage',
                      '8_File-Licence',
                      '9_File-Embargo',
                      '90999_ItemID']]
# cropped_df

In [104]:
# Datumsspalte bereinigen, da durch Excel-Export verändert und Werte auffüllen
cropped_df['Datum'] = cropped_df['Datum'].astype(str).str.extract(r'(\d{4})')
cropped_df['Datum'] = cropped_df['Datum'].ffill() 

# Spalteninhalt wieder als Zahl definieren
cropped_df['Datum'] = cropped_df['Datum'].astype(int)

# Blick auf die enthaltenen Daten 
cropped_df['Datum'].unique()

C:\Users\PC\AppData\Local\Temp\ipykernel_13848\475377001.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cropped_df['Datum'] = cropped_df['Datum'].astype(str).str.extract(r'(\d{4})')
C:\Users\PC\AppData\Local\Temp\ipykernel_13848\475377001.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cropped_df['Datum'] = cropped_df['Datum'].ffill()
C:\Users\PC\AppData\Local\Temp\ipykernel_13848\475377001.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .

array([2026, 2025, 2024, 2023, 2017, 2022, 2020, 2021])

# Hier die gewünschten Jahre für Abfrage eintragen

In [105]:
years = [2023, 2024, 2025]   # oder bei einem Einzahljahr so [2025]

df_matching_years = cropped_df[cropped_df['Datum'].isin(years)].copy()

In [106]:
# Alle Zeilen löschen, die keine File-Informationen haben - das reduziert die Anzahl Titeln entsprechend um alle Titel, die keinen Volltext haben

df_mit_file = df_matching_years
df_mit_file.dropna(subset=['1_File-Description'], inplace=True)  

In [46]:
# Schritt der Reduzierung auf bestimmte Positionen im Status - da wir beschlossen haben Emeriti, Associates, Visiting und Other rauszulassen ???

#desired_statuses = ['Doctoral Student','Postdoc', 'Senior Researcher','Research Group Leader','Director']
#df_final = df[df['Status'].isin(desired_statuses)]

Wie viele Datensätze befinden sich im Embargo und bis zu welchem Datum?

### Was machen wir mit denen?

In [107]:
embargo_counts= df_matching_years['9_File-Embargo'].value_counts()

# Sortiere aufsteigend nach Datum - hier können bestimmte Titel mitgerechnet werden, wenn Embargo vor Zahlenverwertung aufgehoben sein wird
embargo_counts_sorted_asc = embargo_counts.sort_index(ascending=True)


embargo_counts_sorted_asc

9_File-Embargo
2026-04-30    4
2026-06-30    1
2026-07-31    1
2026-08-31    2
2026-09-30    2
2026-10-31    1
2026-12-31    1
2027-02-28    1
Name: count, dtype: int64

# Ab hier Work in Progress

In [ ]:
# Nachzählen, wie sich die Zahl der Titelsätze verändert hat
def count_unique_itemids(df, col='90999_ItemID'):
    return df[col].nunique()

print("Datensätze in der Eingabedatei:", count_unique_itemids(df_pure))

print("Datensätze mit passenden Erscheinungsjahren:", count_unique_itemids())

print("Datensätze mit File-Informationen:", count_unique_itemids(df_mit_file))

df_matching_years.to_excel('Daten/Rohdaten_OA.xlsx', index=False)              # Extrakt für ggf. anderweitige Verwendung


Datensätze in der Eingabedatei: 447
Datensätze mit passenden Erscheinungsjahren: 373
Datensätze mit File-Informationen: 373


In [112]:
# Umbenennung zur weiteren einfacheren Verwendung
df = df_matching_years

In [ ]:
# Orientierungsabfrage für open Access in Beschreibung - ist nicht komplett zuverlässig, wegen Abweichungen im Text, z.B. bei Dissertationen etc.

# Gruppiere nach Jahr und 0999_ItemID, um für jedes Item nur einen Eintrag zu haben
# Priorität: zuerst "Full text open access via publisher", dann "Full text open access" (letztere sind i.d.R. unsere Zweitpublikationen )
# Wir verwenden .first() nach Sortierung, um die Priorität zu gewährleisten

# Sortiere nach Priorität: "Full text open access via publisher" zuerst, dann "Full text open access"
df_sorted = df.sort_values(
    by=['Datum', '90999_ItemID', '1_File-Description'],
    key=lambda x: x.map({
        'Full text open access via publisher': 0,
        'Full text open access': 1,
        # Alle anderen Werte werden als 2 behandelt (höhere Priorität = niedrigere Zahl)
    }).fillna(2)
)

# Entferne Duplikate pro 0999_ItemID, behalte nur den ersten Eintrag (mit höherer Priorität)
df_unique = df_sorted.drop_duplicates(subset=['Datum', '90999_ItemID'], keep='first')

# Zähle nun pro Jahr die Anzahl der verschiedenen Typen
result1 = df_unique.groupby(['Datum', '1_File-Description']).size().unstack(fill_value=0)

result1 = result1.reset_index()

print("Anzahl pro Jahr und Open Access Typ ohne Datenpublikationen (je Item nur einmal):")
result1 

# spalten und Zeilen tauschen für bessere Lesbarkeit der Ergebnisse
df_transponiert = result1.T

# Pro Zeile Berechnung der Summe und enstprechende Sortierung
df_transponiert['Summe'] = df_transponiert.sum(axis=1)
df_sortiert = df_transponiert.sort_values(by='Summe', ascending=False)

# aber dann muss wieder etwas aufgeräumt werden, Indexbereinigung, Spaltentitel korrigieren
df_reindex = df_sortiert.reset_index(drop=False)
df_reindex.columns = df_reindex.iloc[0]
df_new_header = df_reindex.drop(index=0).reset_index(drop=True)

index = df_new_header.columns.get_loc('Datum')
df_new_header.insert(loc=index, column='Text', value=df_new_header['Datum'])
df_new_header.drop(columns='Datum', inplace=True)
df_new_header.columns.values[4] = 'Summe'

full = df_new_header['Text'].str.contains('Full text')
df_fulltext_categories = df_new_header[full]

df_fulltext_categories.to_excel('Results/Allg_Zahlen_Volltextkategorien.xlsx', index=False)
df_fulltext_categories

Anzahl pro Jahr und Open Access Typ ohne Datenpublikationen (je Item nur einmal):


,Text,2023,2024,2025,Summe
0,Full text open access via publisher,32,57,63,152
1,Full text,28,28,29,85
2,Full text open access,31,26,15,72
3,Full text via publisher,11,3,10,24
6,Full text via SocArXiv,4,3,0,7
8,Full text via OSF,0,1,0,1
9,Full text via OSF Preprints,0,1,0,1
10,Full text via SSRN,1,0,0,1


In [ ]:
# Orientierungsabfrage auf Basis der Open Access Einordnungen oder sinnvoll für Export?

df_oa_status = df.sort_values(
    by=['90999_ItemID', '6_File-OA'],
    key=lambda x: x.map({
        'GOLD': 0,
        'GREEN': 1,
        'HYBRID': 2,
        # Alle anderen Werte werden als 3 behandelt (höhere Priorität = niedrigere Zahl)
    }).fillna(3)
)

# Entferne Duplikate pro 0999_ItemID, behalte nur den ersten Eintrag (mit Priorität)
df_oa = df_oa_status.drop_duplicates(subset=['Datum', '90999_ItemID'], keep='first')

# Zähle nun pro Jahr die Anzahl der verschiedenen Typen
result2 = df_oa.groupby(['Datum', '6_File-OA']).size().unstack(fill_value=0)


result2

# Tabelle rechnet richtig, Gesamtmenge 224!

6_File-OA,CLOSED_ACCESS,GOLD,GREEN,HYBRID,MISCELLANEOUS,NOT_SPECIFIED
Datum,,,,,,
2023,4,9,33,7,1,49
2024,3,15,43,12,2,35
2025,10,38,26,9,2,16


Was haben wir für Kategorien, Genre, Kombinationen?   

1. Data_Publication > hier nie ein OA-Status (alle unspecified) + Public visibility - reicht also einfaches zählen für die Jahre oder vorsichtshalber noch die 3_Kategorie hinzunehmen: research-data (sonst kriegt man die Links zum Text dazu)
2. MPIfG Discussion Paper > Suche in 1101_Quellentitel nach "MPIfG Discussion Paper" , hier uneinheitlich OA-Einordnung, vor 2025 unspecified, aber auch darüber hinaus, alle Full text open Access in Beschreibung
3. Thesis - kraut und Rüben der Zugänglichkeiten, hier in Beschreibung, z.T. Full text open access aber auch Full text via (Uni-repos: KUPs, UniDUE), eigentlich sollten alle any-fulltext sein, dann könnte man über Zugangstatus zählen
4. Articles: Genre (oder brauchen wir "MPIfG_JA"?), hier Beschreibung "Full text..." (= closed), "Full text open access ...." (und dann z.T. noch OA-Info), wenn man auf Publisher-version begrenzt und dann den OA-Status zählt (weil es ja um Verlagstexte geht, also ist status hier relevant.)? Hier gibt es auch zwei OA Pre-prints. 
in zweiter Stufe dann schauen, was wie es mit den hochgeladenen aussieht. Also ein Unterset, wo es keinen freien Volltext gibt (d.h. drop alle ID, bei denen e seine Full text open access via publisher gibt)

# Hier Experimente mit abgewandeltem Skript von GLP

In [243]:
# 📌 Centralized Genre Definitions
genre_config = {
    'book': {
        'genres': ['BOOK'],
        'description': 'Books'
    },
    'article': {
        'genres': ['ARTICLE', 'BOOK_REVIEW', 'EDITORIAL'],
        'description': 'Journal Articles'
    },
    'book_contributions': {
        'genres': [
            'CONTRIBUTION_TO_COLLECTED_EDITION',
            'CONTRIBUTION_TO_ENCYCLOPEDIA',
            'CONTRIBUTION_TO_HANDBOOK',
            'BOOK_ITEM'
        ],
        'description': 'Book Contributions'
    },
    'data': {
        'genres': ['DATA_PUBLICATION'],
        'description': 'Data Publications'
    },
    'thesis': {
        'genres': ['THESIS'],
        'description': 'Theses'
    },
    'DP': {
        'genres': ['PAPER'],
        'description': 'MPIfG Discussion Papers'
    },
    'others': {
        'genres': ['PAPER'],
        'description': 'Others'
    }
}

In [244]:
# String wird zur Reduktion auf Volltexte verwendet, da wir das Konsequent vergeben


# 📌 Define the exact string, so only data with full-texts are taken into account
OA_PHRASE = "Full text"

In [245]:
# Datenauswahl: Nur die Items mit einem File, dann aber auch nur die mit öffentlichem Text (Reduktion auf 5_File-visibility: public), 
# Problematisch sind hier die unterschiedlichen Texte, die man gar nicht so einfach auswerten kann, wir müssten vielleicht etwas umschreiben in Other Full text?? dann aber in Kombi mit Public

visibility = ['PUBLIC']   # aber da fallen dann auch die mit Embargo raus, vielleicht "mechanisch ändern?"

df = df_mit_file[df_mit_file['5_File-visibility'].isin(visibility)].copy()


In [246]:
for genre_name, config in genre_config.items():
    genre_list = config['genres']
    
    for year in years:
        print(f"\n{'='*60}")
        print(f"Processing: {genre_name.upper()} ({config['description']}) | Year: {year}")
        print(f"{'='*60}")

        # Normalize inputs
        if isinstance(genre_list, str):
            genre_list = [genre_list]
        if isinstance(year, (int,)):
            year_list = [str(year)]
        elif isinstance(year, str):
            year_list = [year]
        else:
            year_list = [str(year)]

        # 📌 Base filtering: Genre, Year, AND Full text (applies to ALL genres)
        df_genre = df[df['1002_Genre'].isin(genre_list)].copy()
        print(f"  Debug: df_genre shape after genre filter: {df_genre.shape}")
        print(f"  Debug: year_list: {year_list}")
        if 'Datum' in df_genre.columns:
            print(f"  Debug: 'Datum' column dtype: {df_genre['Datum'].dtype}")
            print(f"  Debug: Unique values in 'Datum' column: {df_genre['Datum'].unique()}")

            # Convert 'Datum' to year string for comparison
            if pd.api.types.is_datetime64_any_dtype(df_genre['Datum']):
                df_genre['Datum_Year_Str'] = df_genre['Datum'].dt.year.astype(str)
            elif pd.api.types.is_numeric_dtype(df_genre['Datum']):
                df_genre['Datum_Year_Str'] = df_genre['Datum'].astype(str)
            else: # Assume it's a string-like object, try to extract first 4 chars
                df_genre['Datum_Year_Str'] = df_genre['Datum'].astype(str).str[:4]
            
            df_genre_year = df_genre[df_genre['Datum_Year_Str'].isin(year_list)].copy()
            df_genre = df_genre.drop(columns=['Datum_Year_Str'])
        else:
            print("  Debug: 'Datum' column not found in df_genre. Creating empty df_genre_year.")
            df_genre_year = pd.DataFrame()

        print(f"  Debug: df_genre_year shape after year filter (before file filter): {df_genre_year.shape}")
        if '1_File-Description' in df_genre_year.columns:
            print(f"  Debug: Unique values in '1_File-Description' column (before filter): {df_genre_year['1_File-Description'].unique()}")
        else:
            print("  Debug: '1_File-Description' column not found in df_genre_year.")
        print(f"  Debug: OA_PHRASE: {OA_PHRASE}")

        if not df_genre_year.empty and '1_File-Description' in df_genre_year.columns:
            mpi_filter = df_genre_year['1_File-Description'].str.contains(OA_PHRASE, na=False, case=False)
            df_genre_year = df_genre_year[mpi_filter].copy()
        else:
            # If df_genre_year is already empty or no 'Affiliation' column, no need to filter further
            pass
        print(f"  ✅ Filtered by full text file: {df_genre_year.shape[0]} entries with '{OA_PHRASE}'")

        # 🔍 Conditional: Only for DP → filter for MPIfG Discussion Paper
        if genre_name == "DP":
            if not df_genre_year.empty and '1101_Quellentitel' in df_genre_year.columns:
                dp_filter = df_genre_year['1101_Quellentitel'].str.contains('MPIfG Discussion Paper', na=False)
                df_genre_year = df_genre_year[dp_filter].copy()
            print(f"  ✅ Filtered DP: {df_genre_year.shape[0]} entries with 'MPIfG Discussion Paper'")

        # 🔍 Conditional: Only for 'other' → add non-MPIfG PAPER entries
        elif genre_name == "other":
            if not df_genre_year.empty and '1101_Quellentitel' in df_genre_year.columns:
                paper_filter = (
                    (df_genre_year['Genre'] == 'PAPER') &
                    (~df_genre_year['1101_Quellentitel'].str.contains('MPIfG Discussion Paper', na=False))
                )
                additional_papers = df_genre_year[paper_filter].copy()
                if not additional_papers.empty:
                    print(f"  ✅ Added {additional_papers.shape[0]} non-MPIfG PAPER entries")
                    df_genre_year = pd.concat([df_genre_year, additional_papers], ignore_index=True)
                else:
                    print("  ❌ No non-MPIfG PAPER entries to add")
            else:
                 print("  ❌ No non-MPIfG PAPER entries to add (df_genre_year empty or missing '1101_Quellentitel')")

        # ✅ Create variable name
        var_name = f"df_genre_{year}_{genre_name}"
        globals()[var_name] = df_genre_year.copy()

        # ✅ Print summary
        print(f"  Final shape: {df_genre_year.shape}")
        print(f"  Saved to: {var_name}")


Processing: BOOK (Books) | Year: 2023
  Debug: df_genre shape after genre filter: (12, 14)
  Debug: year_list: ['2023']
  Debug: 'Datum' column dtype: int64
  Debug: Unique values in 'Datum' column: [2025 2023 2024]
  Debug: df_genre_year shape after year filter (before file filter): (5, 15)
  Debug: Unique values in '1_File-Description' column (before filter): ['Traducción de: Streeck, Wolfgang (2021). Zwischen Globalismus und Demokratie: Politische Ökonomie im ausgehenden Neoliberalismus. Berlin: Suhrkamp.'
 'Traduction de: Streeck, Wolfgang (2021). Zwischen Globalismus und Demokratie: Politische Ökonomie im ausgehenden Neoliberalismus. Berlin: Suhrkamp.'
 'Full text via publisher']
  Debug: OA_PHRASE: Full text
  ✅ Filtered by full text file: 3 entries with 'Full text'
  Final shape: (3, 15)
  Saved to: df_genre_2023_book

Processing: BOOK (Books) | Year: 2024
  Debug: df_genre shape after genre filter: (12, 14)
  Debug: year_list: ['2024']
  Debug: 'Datum' column dtype: int64
  De

In [248]:
# kleine Extraschleife für die Datenveröffentlichungen, um identisches Frame zu haben, dass in die Endberechnung mit einfliessen kann
genre_config2 = {
    'data': {
        'genres': ['DATA_PUBLICATION'],
        'description': 'Data Publications'
    },
}

# String für Daten in 3_File-Category, damit hier nur die Links zu den Daten ausgewertet werden
DATA_PHRASE = "research-data"
##

for genre_name, config2 in genre_config2.items():
    genre_data = config2['genres']
    
    for year in years:
        print(f"\n{'='*60}")
        print(f"Processing: {genre_name.upper()} ({config2['description']}) | Year: {year}")
        print(f"{'='*60}")

        # Normalize inputs
        if isinstance(genre_data, str):
            genre_data = [genre_data]
        if isinstance(year, (int,)):
            year_list = [str(year)]
        elif isinstance(year, str):
            year_list = [year]
        else:
            year_list = [str(year)]

        # 📌 Base filtering: Genre, Year, AND Full text (applies to ALL genres)
        df_genre = df[df['1002_Genre'].isin(genre_data)].copy()
        print(f"  Debug: df_genre shape after genre filter: {df_genre.shape}")
        print(f"  Debug: year_list: {year_list}")
        if 'Datum' in df_genre.columns:
            print(f"  Debug: 'Datum' column dtype: {df_genre['Datum'].dtype}")
            print(f"  Debug: Unique values in 'Datum' column: {df_genre['Datum'].unique()}")

            # Convert 'Datum' to year string for comparison
            if pd.api.types.is_datetime64_any_dtype(df_genre['Datum']):
                df_genre['Datum_Year_Str'] = df_genre['Datum'].dt.year.astype(str)
            elif pd.api.types.is_numeric_dtype(df_genre['Datum']):
                df_genre['Datum_Year_Str'] = df_genre['Datum'].astype(str)
            else: # Assume it's a string-like object, try to extract first 4 chars
                df_genre['Datum_Year_Str'] = df_genre['Datum'].astype(str).str[:4]
            
            df_genre_year = df_genre[df_genre['Datum_Year_Str'].isin(year_list)].copy()
            df_genre = df_genre.drop(columns=['Datum_Year_Str'])
        else:
            print("  Debug: 'Datum' column not found in df_genre. Creating empty df_genre_year.")
            df_genre_year = pd.DataFrame()

        print(f"  Debug: df_genre_year shape after year filter (before file filter): {df_genre_year.shape}")
        if '3_File-Category' in df_genre_year.columns:
            print(f"  Debug: Unique values in '3_File-Category' column (before filter): {df_genre_year['3_File-Category'].unique()}")
        else:
            print("  Debug: '3_File-Category' column not found in df_genre_year.")
        print(f"  Debug: DATA_PHRASE: {DATA_PHRASE}")

        if not df_genre_year.empty and '3_File-Category' in df_genre_year.columns:
            mpi_filter = df_genre_year['3_File-Category'].str.contains(DATA_PHRASE, na=False, case=False)
            df_genre_year = df_genre_year[mpi_filter].copy()
        else:
            # If df_genre_year is already empty or no 'Affiliation' column, no need to filter further
            pass
        print(f"  ✅ Filtered by full text file: {df_genre_year.shape[0]} entries with '{DATA_PHRASE}'")


        # ✅ Create variable name
        var_name = f"df_genre_{year}_{genre_name}"
        globals()[var_name] = df_genre_year.copy()

        # ✅ Print summary
        print(f"  Final shape: {df_genre_year.shape}")
        print(f"  Saved to: {var_name}")



Processing: DATA (Data Publications) | Year: 2023
  Debug: df_genre shape after genre filter: (41, 14)
  Debug: year_list: ['2023']
  Debug: 'Datum' column dtype: int64
  Debug: Unique values in 'Datum' column: [2024 2025 2023]
  Debug: df_genre_year shape after year filter (before file filter): (12, 15)
  Debug: Unique values in '3_File-Category' column (before filter): ['research-data' 'supplementary-material']
  Debug: DATA_PHRASE: research-data
  ✅ Filtered by full text file: 6 entries with 'research-data'
  Final shape: (6, 15)
  Saved to: df_genre_2023_data

Processing: DATA (Data Publications) | Year: 2024
  Debug: df_genre shape after genre filter: (41, 14)
  Debug: year_list: ['2024']
  Debug: 'Datum' column dtype: int64
  Debug: Unique values in 'Datum' column: [2024 2025 2023]
  Debug: df_genre_year shape after year filter (before file filter): (16, 15)
  Debug: Unique values in '3_File-Category' column (before filter): ['research-data' 'supplementary-material']
  Debug: DA

In [265]:
# Zusammenfügen der einzelnen Auswertungen zu einer Tabelle

# 📌 Step 1: Collect all genre-year DataFrames from globals()
matched_dfs = {
    name: value for name, value in globals().items()
    if name.startswith("df_genre_") and isinstance(value, pd.DataFrame)
}

print(f"✅ Found {len(matched_dfs)} genre-year DataFrames.")

# 📌 Step 2: Prepare the summary list
summary_data = []

# 📌 Step 3: Loop through each DataFrame and extract summary stats
for var_name, df in matched_dfs.items():
    print(f"\n🔍 Processing: {var_name}")

    # Extract year and genre from variable name: e.g., df_genre_2025_book
    parts = var_name.replace("df_genre_", "").split("_")
    if len(parts) < 2:
        print(f"  ❌ Skipping malformed name: {var_name}")
        continue
    year = parts[0]
    genre_name = "_".join(parts[1:])

    # Get description from config
    description = genre_config.get(genre_name, {}).get('description', 'Unknown')

    # 🔍 ✅ Step 3.3: Count file values
    file_counts = df['1_File-Description'].value_counts().to_dict()
    fulltext = file_counts.get('Full text', 0)
    fulltext_publisher = file_counts.get('Full text open access via publisher', 0)
    fulltext_open = file_counts.get('Full text open access', 0)
    fulltext_closed = file_counts.get('Full text via publisher', 0)

    # 🔍 ✅ Step 3.4: Total count
    total = len(df)

    # 🔍 ✅ Step 3.5: Calculate percentages (safe division)
    fulltext_pct = (fulltext / total) * 100 if total > 0 else 0
    fulltext_publisher_pct = (fulltext_publisher / total) * 100 if total > 0 else 0
    fulltext_open_pct = (fulltext_open / total) * 100 if total > 0 else 0 
    fulltext_closed_pct = (fulltext_closed / total ) *100 if total > 0 else 0

    # 🔍 ✅ Step 3.6: Print debug info
    print(f"  ✅ Total: {total}, Fulltext: {fulltext}, Fulltext via Publisher: {fulltext_publisher}, Fulltext open: {fulltext_open}, Fulltext closed: {fulltext_closed}")
    print(f"  ✅ Percentages: {fulltext_pct:.1f}%, {fulltext_publisher_pct:.1f}%, {fulltext_open_pct:.1f}%, {fulltext_closed_pct:.1f}%")

    # Append to summary
    summary_data.append({
        'Year': year,
        'Genre': genre_name,
        'Description': description,
        'Total': total,
        'Fulltext': fulltext,
        'Fulltext_publisher': fulltext_publisher,
        'Fulltext_open': fulltext_open,
        'Fulltext_closed': fulltext_closed,
        'Fullext %': round(fulltext_pct, 1),
        'Fulltext_publisher %': round(fulltext_publisher_pct, 1),
        'Fulltext_open %': round(fulltext_open_pct, 1),
        'Fulltext_closed %': round(fulltext_closed_pct, 1),
    })

# 📌 Step 4: Create the master summary DataFrame
master_summary = pd.DataFrame(summary_data)

# 📌 Step 5: Sort and reset index
master_summary = master_summary.sort_values(['Year', 'Genre']).reset_index(drop=True)

# 📌 Step 6: Add yearly totals (with proper rounding)
yearly_totals = master_summary.groupby('Year', as_index=False)[['Total', 'Fulltext', 'Fulltext_publisher', 'Fulltext_open', 'Fulltext_closed']].sum()
yearly_totals['Genre'] = 'Total'
yearly_totals['Description'] = 'Yearly Total'
yearly_totals['Missing'] = 0
yearly_totals['Fulltext %'] = (yearly_totals['Fulltext'] / yearly_totals['Total']) * 100
yearly_totals['Fulltext_publisher %'] = (yearly_totals['Fulltext_publisher'] / yearly_totals['Total']) * 100
yearly_totals['Fulltext_open %'] = (yearly_totals['Fulltext_open'] / yearly_totals['Total']) * 100
yearly_totals['Fulltext_closed %'] = (yearly_totals['Fulltext_closed'] / yearly_totals['Total']) * 100
yearly_totals = yearly_totals.round(1)

# Append totals
master_summary = pd.concat([master_summary, yearly_totals], ignore_index=True)

✅ Found 22 genre-year DataFrames.

🔍 Processing: df_genre_year
  ❌ Skipping malformed name: df_genre_year

🔍 Processing: df_genre_2023_book
  ✅ Total: 3, Fulltext: 0, Fulltext via Publisher: 0, Fulltext open: 0, Fulltext closed: 3
  ✅ Percentages: 0.0%, 0.0%, 0.0%, 100.0%

🔍 Processing: df_genre_2024_book
  ✅ Total: 2, Fulltext: 0, Fulltext via Publisher: 0, Fulltext open: 0, Fulltext closed: 1
  ✅ Percentages: 0.0%, 0.0%, 0.0%, 50.0%

🔍 Processing: df_genre_2025_book
  ✅ Total: 3, Fulltext: 0, Fulltext via Publisher: 0, Fulltext open: 0, Fulltext closed: 3
  ✅ Percentages: 0.0%, 0.0%, 0.0%, 100.0%

🔍 Processing: df_genre_2023_article
  ✅ Total: 100, Fulltext: 1, Fulltext via Publisher: 31, Fulltext open: 44, Fulltext closed: 23
  ✅ Percentages: 1.0%, 31.0%, 44.0%, 23.0%

🔍 Processing: df_genre_2024_article
  ✅ Total: 119, Fulltext: 0, Fulltext via Publisher: 48, Fulltext open: 52, Fulltext closed: 19
  ✅ Percentages: 0.0%, 40.3%, 43.7%, 16.0%

🔍 Processing: df_genre_2025_article
  ✅ T

In [266]:
# Notwendige Korrekturen für Datenpublikationen, sie haben andere Terminologie und werden darum nicht automatisch oben gezählt
# Erstens Eintrag der Gesamtzahl als Fulltext_open und zweitens der entsprechende Prozenteintrag
master_summary.loc[master_summary['Genre'] == 'data', 'Fulltext_open'] = master_summary.loc[master_summary['Genre'] == 'data', 'Total'] 
master_summary.loc[master_summary['Genre'] == 'data', 'Fulltext_open %'] = 100.0

In [268]:
master_summary

,Year,Genre,Description,Total,Fulltext,Fulltext_publisher,Fulltext_open,Fulltext_closed,Fullext %,Fulltext_publisher %,Fulltext_open %,Fulltext_closed %,Missing,Fulltext %
0,2023,DP,MPIfG Discussion Papers,6,0,0,6,0,0.0,0.0,100.0,0.0,NaN,NaN
1,2023,article,Journal Articles,100,1,31,44,23,1.0,31.0,44.0,23.0,NaN,NaN
2,2023,book,Books,3,0,0,0,3,0.0,0.0,0.0,100.0,NaN,NaN
3,2023,book_contributions,Book Contributions,8,0,1,0,7,0.0,12.5,0.0,87.5,NaN,NaN
4,2023,data,Data Publications,6,0,0,6,0,0.0,0.0,100.0,0.0,NaN,NaN
5,2023,others,Others,14,3,0,7,3,21.4,0.0,50.0,21.4,NaN,NaN
6,2023,thesis,Theses,10,0,0,5,0,0.0,0.0,50.0,0.0,NaN,NaN
7,2024,DP,MPIfG Discussion Papers,9,0,0,9,0,0.0,0.0,100.0,0.0,NaN,NaN
8,2024,article,Journal Articles,119,0,48,52,19,0.0,40.3,43.7,16.0,NaN,NaN
9,2024,book,Books,2,0,0,0,1,0.0,0.0,0.0,50.0,NaN,NaN


In [264]:
# Umsortierung und Vereinfachung der Namen, sowie Export zu Excel

master_summary_sorted = master_summary[['Year',
                           'Description',
                           'Total',
                           'Fulltext_publisher',
                           'Fulltext_publisher %',
                           'Fulltext_open',
                           'Fulltext_open %',
                           'Fulltext',
                           'Fulltext %',  # Den Wert berechnet er nicht richtig
                           'Fulltext_closed',
                           'Fulltext_closed %',]]



open_access_final = master_summary_sorted.rename(columns={    #Vereinfachung der Spaltennamen
    'Description' : 'Genre',
    'Fulltext_publisher' : 'Open Access',
    'Fulltext_publisher %' : '%',
    'Fulltext_open':'Repository Open Access',  #ist das Zutreffend? Prüfen!!!
    'Fulltext_open %': '%',
    'Fulltext':'Full text available',
    'Fulltext %':'%',
    'Fulltext_closed':'Closed Access',
    'Fulltext_closed %': '%'}) 

open_access_final.to_excel('Results/Open_Access_Übersicht.xlsx', index=False)
open_access_final

,Year,Genre,Total,Open Access,%,Repository Open Access,%,Full text available,%,Closed Access,%
0,2023,MPIfG Discussion Papers,6,0,0.0,6,100.0,0,NaN,0,0.0
1,2023,Journal Articles,100,31,31.0,44,44.0,1,NaN,23,23.0
2,2023,Books,3,0,0.0,0,0.0,0,NaN,3,100.0
3,2023,Book Contributions,8,1,12.5,0,0.0,0,NaN,7,87.5
4,2023,Data Publications,6,0,0.0,6,100.0,0,NaN,0,0.0
5,2023,Others,14,0,0.0,7,50.0,3,NaN,3,21.4
6,2023,Theses,10,0,0.0,5,50.0,0,NaN,0,0.0
7,2024,MPIfG Discussion Papers,9,0,0.0,9,100.0,0,NaN,0,0.0
8,2024,Journal Articles,119,48,40.3,52,43.7,0,NaN,19,16.0
9,2024,Books,2,0,0.0,0,0.0,0,NaN,1,50.0
